# 02 — Data Cleaning

Apply the business-scope and data-quality decisions established during the data audit.

The primary KPI population represents **booked GMV**: orders with a valid commercial status and a positive payment value during the complete analysis period.

In [0]:
from pyspark.sql import functions as F

RAW_PATH = "/Volumes/workspace/olist/raw"

In [0]:
orders = spark.read.csv(
    f"{RAW_PATH}/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

order_items = spark.read.csv(
    f"{RAW_PATH}/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

order_payments = spark.read.csv(
    f"{RAW_PATH}/olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)

customers = spark.read.csv(
    f"{RAW_PATH}/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

products = spark.read.csv(
    f"{RAW_PATH}/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

sellers = spark.read.csv(
    f"{RAW_PATH}/olist_sellers_dataset.csv",
    header=True,
    inferSchema=True
)

In [0]:
ANALYSIS_START = "2017-01-01"
ANALYSIS_END = "2018-09-01"

VALID_STATUSES = [
    "delivered",
    "shipped",
    "invoiced",
    "processing",
    "approved"
]

In [0]:
orders_clean = (
    orders
    .filter(
        F.col("order_purchase_timestamp") >=
        F.lit(ANALYSIS_START).cast("timestamp")
    )
    .filter(
        F.col("order_purchase_timestamp") <
        F.lit(ANALYSIS_END).cast("timestamp")
    )
    .filter(
        F.col("order_status").isin(VALID_STATUSES)
    )
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at"
    )
)

In [0]:
display(
    orders_clean.agg(
        F.count("*").alias("rows"),
        F.countDistinct("order_id").alias("unique_order_ids"),
        F.min("order_purchase_timestamp").alias("first_purchase"),
        F.max("order_purchase_timestamp").alias("last_purchase")
    )
)

display(
    orders_clean
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

rows,unique_order_ids,first_purchase,last_purchase
97905,97905,2017-01-05T11:56:06.000Z,2018-08-29T15:00:37.000Z


order_status,count
delivered,96211
shipped,1097
processing,299
invoiced,296
approved,2


## Clean payments

Keep payment records belonging to the scoped order population and retain only positive, non-null payment values.

In [0]:
order_payments_clean = (
    order_payments
    .join(
        orders_clean.select("order_id"),
        on="order_id",
        how="inner"
    )
    .filter(
        F.col("payment_value").isNotNull()
    )
    .filter(
        F.col("payment_value") > 0
    )
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    )
)

In [0]:
display(
    order_payments_clean.agg(
        F.count("*").alias("payment_rows"),
        
        F.countDistinct(
            "order_id",
            "payment_sequential"
        ).alias("unique_payment_keys"),
        
        F.countDistinct(
            "order_id"
        ).alias("orders_with_payment"),
        
        F.sum(
            F.col("payment_value").isNull().cast("int")
        ).alias("missing_payment_value"),
        
        F.sum(
            (F.col("payment_value") <= 0).cast("int")
        ).alias("non_positive_payment")
    )
)

payment_rows,unique_payment_keys,orders_with_payment,missing_payment_value,non_positive_payment
102248,102248,97905,0,0


In [0]:
payment_sequence_issues = (
    order_payments_clean
    .groupBy("order_id")
    .agg(
        F.count("*").alias("payment_row_count"),
        F.min("payment_sequential").alias("min_payment_sequence"),
        F.max("payment_sequential").alias("max_payment_sequence")
    )
    .filter(
        (F.col("min_payment_sequence") != 1) |
        (
            F.col("max_payment_sequence") !=
            F.col("payment_row_count")
        )
    )
)

display(payment_sequence_issues)

order_id,payment_row_count,min_payment_sequence,max_payment_sequence
d1329f2d9a7c9c972b1ab5c6b7c8d3a5,1,2,2
1896e8b8fd196151559b02e31e98ce89,1,2,2
a6d23aa5f1190c09129345b363de8e37,1,2,2
d2c902f172e80f22126d53d9ff7871b2,1,2,2
cfa1591318ed6c901b0c80debfd4b811,1,2,2
2f4597354cca1be196ca6b609711c109,1,2,2
510bff1cf06be1143d3b6698df2fd486,1,2,2
4b2dcf714729a31ae2d3b69b1dd35629,1,2,2
9769f28d910e7304d7bf0fcd32d8ecd6,1,2,2
da9aa61ec7bb8713f483bf0bf94c71c2,1,2,2


In [0]:
display(
    payment_sequence_issues.agg(
        F.count("*").alias("orders_with_sequence_issue")
    )
)

orders_with_sequence_issue
80


In [0]:
sequence_issue_examples = (
    order_payments
    .join(
        payment_sequence_issues.select("order_id"),
        on="order_id",
        how="inner"
    )
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    )
    .orderBy(
        "order_id",
        "payment_sequential"
    )
)

display(
    sequence_issue_examples.limit(50)
)

order_id,payment_sequential,payment_type,payment_installments,payment_value
00ac05fe0fc047c54418098eb64e3aaa,2,debit_card,1,123.47
056c68d093c100017aab1f00f260705c,2,debit_card,1,52.78
0668d086b3ae41adde3aed3dacbc8fae,2,debit_card,1,252.74
0a7d898c6305101e69e9f5a05cd0130d,2,credit_card,5,215.47
0d0e88a418636d40ea780339701db49c,2,debit_card,1,121.09
0fdb6f73fab7005c84d4a43d8a93682b,2,debit_card,1,34.13
159da9914b51de617fe80162939965c5,2,debit_card,1,40.76
166b01d0fce75a595b92ad9e84e0843b,2,credit_card,6,127.09
1896e8b8fd196151559b02e31e98ce89,2,debit_card,1,108.95
1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [0]:
sequence_issue_payment_totals = (
    order_payments
    .join(
        payment_sequence_issues.select("order_id"),
        on="order_id",
        how="inner"
    )
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_total")
    )
)

In [0]:
sequence_issue_item_totals = (
    order_items
    .join(
        payment_sequence_issues.select("order_id"),
        on="order_id",
        how="inner"
    )
    .groupBy("order_id")
    .agg(
        F.sum(
            F.col("price") +
            F.col("freight_value")
        ).alias("item_plus_freight_total")
    )
)

In [0]:
payment_sequence_reconciliation = (
    payment_sequence_issues
    .select("order_id")
    .join(
        sequence_issue_payment_totals,
        on="order_id",
        how="left"
    )
    .join(
        sequence_issue_item_totals,
        on="order_id",
        how="left"
    )
    .withColumn(
        "difference",
        F.round(
            F.col("payment_total") -
            F.col("item_plus_freight_total"),
            2
        )
    )
)

In [0]:
display(
    payment_sequence_reconciliation.agg(
        F.count("*").alias("sequence_issue_orders"),
        
        F.sum(
            (F.col("difference") != 0).cast("int")
        ).alias("orders_with_value_difference"),
        
        F.max(
            F.abs("difference")
        ).alias("max_absolute_difference")
    )
)

sequence_issue_orders,orders_with_value_difference,max_absolute_difference
80,2,0.01


In [0]:
display(
    payment_sequence_reconciliation
    .filter(
        F.col("difference") != 0
    )
    .orderBy(
        F.desc(F.abs("difference"))
    )
)

order_id,payment_total,item_plus_freight_total,difference
1896e8b8fd196151559b02e31e98ce89,108.95,108.96000000000001,-0.01
3263e5243c28ebe706f7500cf88054ad,369.65,369.66,-0.01


### Payment sequence audit decision

- 80 orders have non-contiguous payment sequence numbers.
- The anomaly already exists in the raw source data.
- Payment totals reconcile with item plus freight totals.
- Only 2 orders have a 0.01 BRL rounding difference.
- No orders are excluded and payment sequence values are not modified.

In [0]:
order_items_clean = (
    order_items
    .join(
        orders_clean.select("order_id"),
        on="order_id",
        how="inner"
    )
    .filter(
        F.col("product_id").isNotNull()
    )
    .filter(
        F.col("seller_id").isNotNull()
    )
    .filter(
        F.col("price") >= 0
    )
    .filter(
        F.col("freight_value") >= 0
    )
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    )
)

In [0]:
display(
    order_items_clean.agg(
        F.count("*").alias("item_rows"),
        F.countDistinct(
            "order_id",
            "order_item_id"
        ).alias("unique_item_keys"),
        F.countDistinct(
            "order_id"
        ).alias("orders_with_items"),
        F.sum("price").alias("merchandise_value"),
        F.sum("freight_value").alias("freight_value")
    )
)

item_rows,unique_item_keys,orders_with_items,merchandise_value,freight_value
111752,111752,97905,1.3449529679998713E7,2234177.0600000136


In [0]:
products_clean = (
    products
    .select(
        "product_id",
        "product_category_name"
    )
    .withColumn(
        "product_category_name",
        F.when(
            F.col("product_category_name").isNull() |
            (F.trim(F.col("product_category_name")) == ""),
            F.lit("unknown")
        ).otherwise(
            F.trim(F.col("product_category_name"))
        )
    )
)

In [0]:
display(
    products_clean.agg(
        F.count("*").alias("product_rows"),
        F.countDistinct("product_id").alias("unique_product_ids"),
        F.sum(
            (F.col("product_category_name") == "unknown")
            .cast("int")
        ).alias("products_with_unknown_category")
    )
)

product_rows,unique_product_ids,products_with_unknown_category
32951,32951,610


In [0]:
order_items_enriched = (
    order_items_clean
    .join(
        products_clean,
        on="product_id",
        how="left"
    )
    .withColumn(
        "product_category_name",
        F.coalesce(
            F.col("product_category_name"),
            F.lit("unknown")
        )
    )
)

In [0]:
display(
    order_items_enriched.agg(
        F.count("*").alias("item_rows_after_join"),
        F.countDistinct(
            "order_id",
            "order_item_id"
        ).alias("unique_item_keys"),
        F.sum(
            (F.col("product_category_name") == "unknown")
            .cast("int")
        ).alias("unknown_category_items")
    )
)

item_rows_after_join,unique_item_keys,unknown_category_items
111752,111752,1587


In [0]:
customers_clean = (
    customers
    .join(
        orders_clean
        .select("customer_id")
        .distinct(),
        on="customer_id",
        how="inner"
    )
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
    .withColumn(
        "customer_city",
        F.lower(F.trim("customer_city"))
    )
    .withColumn(
        "customer_state",
        F.upper(F.trim("customer_state"))
    )
)

In [0]:
sellers_clean = (
    sellers
    .join(
        order_items_clean
        .select("seller_id")
        .distinct(),
        on="seller_id",
        how="inner"
    )
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    )
    .withColumn(
        "seller_city",
        F.lower(F.trim("seller_city"))
    )
    .withColumn(
        "seller_state",
        F.upper(F.trim("seller_state"))
    )
)

In [0]:
display(
    customers_clean.agg(
        F.count("*").alias("customer_rows"),
        F.countDistinct("customer_id").alias("unique_customer_ids"),
        F.sum(
            F.col("customer_state")
            .isNull()
            .cast("int")
        ).alias("missing_customer_state")
    )
)

customer_rows,unique_customer_ids,missing_customer_state
97905,97905,0


In [0]:
display(
    sellers_clean.agg(
        F.count("*").alias("seller_rows"),
        F.countDistinct("seller_id").alias("unique_seller_ids"),
        F.sum(
            F.col("seller_state")
            .isNull()
            .cast("int")
        ).alias("missing_seller_state")
    )
)

seller_rows,unique_seller_ids,missing_seller_state
3029,3029,0


## Persist clean source tables

The cleaned source-level datasets are saved as Delta tables.

Currency values are kept at their original precision. Rounding will only be applied when presenting analytical results.

In [0]:
clean_tables = {
    "clean_orders": orders_clean,
    "clean_order_payments": order_payments_clean,
    "clean_order_items": order_items_clean,
    "clean_products": products_clean,
    "clean_customers": customers_clean,
    "clean_sellers": sellers_clean
}

for table_name, dataframe in clean_tables.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"workspace.olist.{table_name}"
        )
    )

In [0]:
display(
    spark.sql(
        "SHOW TABLES IN workspace.olist"
    )
    .filter(
        F.col("tableName").startswith("clean_")
    )
    .orderBy("tableName")
)

database,tableName,isTemporary
olist,clean_customers,false
olist,clean_order_items,false
olist,clean_order_payments,false
olist,clean_orders,false
olist,clean_products,false
olist,clean_sellers,false
